In [13]:
import pandas as pd
import numpy as np
import random
import os
import time
from datetime import datetime

# Import Google GenAI SDK terbaru
from google import genai
from google.genai import types


In [14]:
# Konfigurasi Master Data
MESIN_LIST = [
    'Pompa Sentrifugal A', 
    'Kompresor Gas B', 
    'Turbin Uap C', 
    'Motor Induksi D', 
    'Sistem Konveyor E'
]

INTERVAL_KATEGORI = ['Harian', 'Mingguan', 'Bulanan', 'Triwulan', 'Semester', 'Tahunan']

In [15]:
def generate_prediction_inputs(num_samples=100):
    """
    Menghasilkan data dummy berupa output dari 3 model AI utama Exigen.
    """
    data = []
    for _ in range(num_samples):
        mesin = random.choice(MESIN_LIST)
        
        # 1. Output dari Maintenance Predictor
        rul_days = random.randint(0, 400) # Remaining Useful Life
        prob_kegagalan = round(random.uniform(0.01, 0.99), 2)
        
        # 2. Output dari Cost Estimator
        # Semakin tinggi probabilitas kegagalan, biasanya potensi biaya kerusakan semakin besar
        base_cost = random.randint(500000, 5000000)
        multiplier = 1 + (prob_kegagalan * 10)
        estimasi_biaya = int(base_cost * multiplier)
        
        # 3. Output dari Interval Classifier
        if rul_days < 7: interval = 'Harian'
        elif rul_days < 30: interval = 'Mingguan'
        elif rul_days < 90: interval = 'Bulanan'
        elif rul_days < 180: interval = 'Triwulan'
        elif rul_days < 365: interval = 'Semester'
        else: interval = 'Tahunan'
            
        data.append({
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'nama_mesin': mesin,
            'rul_days': rul_days,
            'probabilitas_kegagalan': prob_kegagalan,
            'estimasi_biaya': estimasi_biaya,
            'interval_pemeliharaan': interval
        })
        
    return pd.DataFrame(data)
def generate_prediction_inputs(num_samples=100):
    """
    Menghasilkan data dummy berupa output dari 3 model AI utama Exigen.
    """
    data = []
    for _ in range(num_samples):
        mesin = random.choice(MESIN_LIST)
        
        # 1. Output dari Maintenance Predictor
        rul_days = random.randint(0, 400) # Remaining Useful Life
        prob_kegagalan = round(random.uniform(0.01, 0.99), 2)
        
        # 2. Output dari Cost Estimator
        # Semakin tinggi probabilitas kegagalan, biasanya potensi biaya kerusakan semakin besar
        base_cost = random.randint(500000, 5000000)
        multiplier = 1 + (prob_kegagalan * 10)
        estimasi_biaya = int(base_cost * multiplier)
        
        # 3. Output dari Interval Classifier
        if rul_days < 7: interval = 'Harian'
        elif rul_days < 30: interval = 'Mingguan'
        elif rul_days < 90: interval = 'Bulanan'
        elif rul_days < 180: interval = 'Triwulan'
        elif rul_days < 365: interval = 'Semester'
        else: interval = 'Tahunan'
            
        data.append({
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'nama_mesin': mesin,
            'rul_days': rul_days,
            'probabilitas_kegagalan': prob_kegagalan,
            'estimasi_biaya': estimasi_biaya,
            'interval_pemeliharaan': interval
        })
        
    return pd.DataFrame(data)

In [18]:
# Inisialisasi client Google GenAI
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

client = genai.Client()

def generate_narrative_report_gemini(row):
    """
    Mengubah baris data hasil prediksi menjadi paragraf laporan naratif 
    menggunakan model Generative AI Gemini.
    """
    mesin = row['nama_mesin']
    rul = row['rul_days']
    prob = row['probabilitas_kegagalan'] * 100
    biaya_format = f"Rp {row['estimasi_biaya']:,}".replace(',', '.')
    interval = row['interval_pemeliharaan'].lower()
    
    # Prompt Engineering: Instruksi spesifik agar Gemini paham konteksnya
    prompt = f"""
    Kamu adalah asisten AI 'Exigen Predictive Maintenance' yang ahli dalam membuat laporan kesehatan aset industri.
    Buatkan 1 paragraf laporan naratif yang natural, profesional, dan mudah dipahami teknisi mengenai kesehatan aset berikut:
    
    - Nama Mesin: {mesin}
    - Sisa Umur Pakai (Remaining Useful Life): {rul} hari
    - Probabilitas Kegagalan Anomali: {prob:.0f}%
    - Estimasi Biaya Perbaikan jika rusak fatal: {biaya_format}
    - Rekomendasi Interval Pemeliharaan: {interval}
    
    Aturan Penulisan:
    1. Laporan harus berisi analisis status saat ini, seberapa kritis kondisinya, peringatan biaya, dan rekomendasi tindakan.
    2. Tulis dalam bentuk 1 paragraf saja (sekitar 3-5 kalimat).
    3. Jangan gunakan format poin-poin (bullet points).
    4. Gunakan bahasa Indonesia baku yang profesional.
    """
    
    try:
        # Memanggil model gemini (gemini-2.5-flash direkomendasikan untuk teks cepat & murah)
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.7, # Sedikit kreativitas agar teks tidak kaku
            )
        )
        return response.text.strip()
    except Exception as e:
        return f"Gagal menghasilkan laporan AI: {str(e)}"


In [19]:
# Tentukan jumlah sampel (Ubah jadi 5 atau 10 dulu untuk testing agar aman dari limit API)
NUM_SAMPLES = 5

print("1. Membuat data prediksi dummy dari model...")
df_maintenance = generate_prediction_inputs(num_samples=NUM_SAMPLES)

print(f"2. Meng-generate {NUM_SAMPLES} teks laporan menggunakan Gemini AI...")
print("   (Proses ini mungkin memakan waktu karena memanggil API satu per satu)")

laporan_list = []
for index, row in df_maintenance.iterrows():
    # Menjalankan AI
    laporan = generate_narrative_report_gemini(row)
    laporan_list.append(laporan)
    
    print(f"   - Selesai men-generate laporan untuk {row['nama_mesin']}...")
    
    # Jeda 2 detik untuk menghindari error 'Too Many Requests (Rate Limit)'
    time.sleep(2)

# Memasukkan hasil ke dalam DataFrame
df_maintenance['laporan_naratif'] = laporan_list

# Atur tampilan pandas agar isi teks terlihat penuh tanpa terpotong
pd.set_option('display.max_colwidth', None)

# Tampilkan hasilnya
display(df_maintenance[['nama_mesin', 'laporan_naratif']])

1. Membuat data prediksi dummy dari model...
2. Meng-generate 5 teks laporan menggunakan Gemini AI...
   (Proses ini mungkin memakan waktu karena memanggil API satu per satu)
   - Selesai men-generate laporan untuk Turbin Uap C...
   - Selesai men-generate laporan untuk Turbin Uap C...
   - Selesai men-generate laporan untuk Turbin Uap C...
   - Selesai men-generate laporan untuk Pompa Sentrifugal A...
   - Selesai men-generate laporan untuk Motor Induksi D...


,nama_mesin,laporan_naratif
0,Turbin Uap C,"Berdasarkan analisis terkini, Turbin Uap C saat ini menunjukkan kondisi yang mengkhawatirkan dengan sisa umur pakai terprediksi hanya 153 hari dan probabilitas kegagalan anomali yang sangat tinggi mencapai 77%. Tingginya risiko ini berpotensi menyebabkan kerusakan fatal yang diperkirakan memerlukan biaya perbaikan signifikan sekitar Rp 11.120.809. Mengingat kondisi kritis tersebut, sangat direkomendasikan untuk segera melakukan inspeksi mendalam dan menjadwalkan tindakan pemeliharaan preventif secara triwulan guna mitigasi risiko serta menjaga keandalan operasional."
1,Turbin Uap C,"Berdasarkan analisis data terkini, Turbin Uap C diperkirakan memiliki sisa umur pakai sekitar 394 hari. Meskipun probabilitas kegagalan anomali saat ini teridentifikasi pada tingkat 10% yang relatif rendah, penting untuk tetap waspada mengingat potensi biaya perbaikan yang dapat mencapai Rp 9.920.902 jika terjadi kerusakan fatal. Oleh karena itu, kami merekomendasikan untuk mempertahankan interval pemeliharaan tahunan yang sudah ada, dengan penekanan pada inspeksi komprehensif untuk deteksi dini potensi anomali guna memastikan operasional yang optimal dan mencegah kerugian finansial yang tidak terduga."
2,Turbin Uap C,"Berdasarkan analisis terkini, Turbin Uap C diperkirakan memiliki sisa umur pakai sekitar 342 hari. Namun, probabilitas kegagalan anomali yang mencapai 31% mengindikasikan risiko operasional yang signifikan, sehingga memerlukan perhatian mendalam untuk mencegah gangguan yang tidak terduga. Perlu diperhatikan bahwa kegagalan fatal dapat memicu estimasi biaya perbaikan hingga Rp 15.415.360. Oleh karena itu, sangat direkomendasikan untuk menerapkan interval pemeliharaan semesteran secara ketat guna menjaga keandalan dan memitigasi potensi kerugian."
3,Pompa Sentrifugal A,"Exigen Predictive Maintenance melaporkan bahwa Pompa Sentrifugal A menunjukkan sisa umur pakai sekitar 353 hari; namun, probabilitas kegagalan anomali mencapai 54%, mengindikasikan adanya risiko operasional yang signifikan dan memerlukan perhatian segera. Apabila terjadi kegagalan fatal, estimasi biaya perbaikan diperkirakan mencapai Rp 19.830.150, menyoroti urgensi tindakan pencegahan untuk menghindari kerugian finansial yang substansial. Oleh karena itu, interval pemeliharaan semesteran yang telah direkomendasikan harus dipatuhi secara ketat, dan kami menyarankan inspeksi lebih mendalam untuk mengidentifikasi akar masalah serta melakukan intervensi proaktif."
4,Motor Induksi D,"Berdasarkan analisis terkini, Motor Induksi D menunjukkan sisa umur pakai sekitar 342 hari; namun, probabilitas kegagalan anomali yang mencapai 56% mengindikasikan adanya peningkatan risiko kerusakan signifikan dalam waktu dekat. Kondisi ini memerlukan perhatian serius mengingat potensi biaya perbaikan fatal yang diperkirakan mencapai Rp 23.682.258 jika kegagalan terjadi. Oleh karena itu, sangat direkomendasikan untuk segera melakukan inspeksi mendalam dan mempertimbangkan jadwal pemeliharaan preventif yang lebih intensif daripada interval semesteran yang ditetapkan, guna memitigasi risiko kegagalan tak terduga dan mencegah kerugian operasional yang substansial."


In [21]:
# Pastikan folder data eksis di level direktori root (sesuai struktur repo)
output_dir = '../../data'
os.makedirs(output_dir, exist_ok=True)

# Nama file
file_path = os.path.join(output_dir, 'synthetic_report_dataset.csv')

# Export ke CSV
df_maintenance.to_csv(file_path, index=False)
print(f"✅ Berhasil! Dataset laporan naratif telah disimpan di: {file_path}")


✅ Berhasil! Dataset laporan naratif telah disimpan di: ../../data\synthetic_report_dataset.csv
